## Data Cleaning & Preprocessing

In this step, we clean the raw transaction dataset to make it suitable for analysis.

### What happens here:
- Load raw transaction data from the `data/raw` directory
- Apply data cleaning logic such as:
  - Handling missing values
  - Fixing data types
  - Removing invalid or inconsistent records
- Save the cleaned dataset to the `data/processed` directory for reuse

This step ensures the dataset is structured, consistent, and ready for downstream analysis such as RFM segmentation or exploratory data analysis.


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loading import load_transactions
from src.preprocessing import clean_transactions, save_cleaned_data

# Load raw data
df_raw = load_transactions("../data/raw/transactions.csv")

# Clean data
df_clean = clean_transactions(df_raw)

# Save cleaned data
save_cleaned_data(df_clean, "../data/processed/cleaned_transactions.csv")

df_clean.info()


## RFM Feature Engineering (Recency, Frequency, Monetary)

In this step, we calculate **RFM metrics** to analyze customer purchasing behavior.

### What is RFM?
- **Recency (R):** How recently a customer made a purchase  
- **Frequency (F):** How often the customer purchases  
- **Monetary (M):** How much the customer spends  

### What happens here:
- Load and clean transaction data
- Aggregate customer-level data
- Compute RFM metrics for each customer
- Prepare the dataset for customer segmentation

This transformation converts raw transaction data into meaningful customer-level features used for segmentation and business insights.


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loading import load_transactions
from src.preprocessing import clean_transactions
from src.rfm_calculation import calculate_rfm

# Load and clean data
df_raw = load_transactions("../data/raw/transactions.csv")
df_clean = clean_transactions(df_raw)

# Calculate RFM
rfm_df = calculate_rfm(df_clean)

rfm_df.head()
rfm_df.info()


## RFM Scoring & Customer Segmentation

In this step, we convert raw RFM values into meaningful customer scores and segments.

### What happens here:
- Calculate RFM metrics for each customer
- Assign scores to Recency, Frequency, and Monetary values
- Combine scores to form customer segments
- Prepare data for customer segmentation and business analysis

### Why this matters:
RFM scoring helps identify:
- High-value customers
- Loyal customers
- At-risk customers
- Low-engagement customers

This allows businesses to target customers with personalized marketing and retention strategies.


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rfm_calculation import calculate_rfm
from src.scoring import score_rfm

rfm_df = calculate_rfm(df_clean)

rfm_scored = score_rfm(rfm_df)

rfm_scored.head()
rfm_scored.info()

## Customer Segmentation & Final Output

In this step, we assign meaningful customer segments based on RFM scores.

### What happens here:
- Apply business rules to classify customers into segments
- Generate human-readable customer labels
- Save the final segmented dataset for reporting and analysis

### Example Segments:
- High Value Customers
- Loyal Customers
- Potential Loyalists
- At-Risk Customers
- Low-Value Customers

### Output:
- Final RFM table with customer segments
- Ready-to-use dataset for dashboards, marketing, and reporting


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.segmentation import label_customers

rfm_labeled = label_customers(rfm_scored)

# Save final table
rfm_labeled.to_csv("../data/processed/rfm_table.csv", index=False)

rfm_labeled["Segment"].value_counts()
rfm_labeled.head()


## Data Visualization & Insights Generation

In this step, we generate visual insights from the final RFM segmentation results.

### What happens here:
- Create visualizations for customer segments
- Analyze distribution of customers across RFM groups
- Generate charts for business interpretation
- Save all plots to the output directory

### Purpose:
These visualizations help stakeholders:
- Understand customer behavior
- Identify high-value and at-risk customers
- Support marketing and retention decisions

All charts are saved automatically for reporting and presentation use.


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.visualization import create_all_plots

create_all_plots(
    rfm_labeled,
    "../outputs/charts"
)


## Business Insights Report Generation

In this step, we generate a **text-based business insights report** from the final RFM segmentation output.

### What happens here:
- Calculate total number of customers
- Compute segment-wise customer distribution
- Calculate average RFM values per segment
- Generate human-readable business insights
- Save the summary as a `.txt` report for stakeholders

### Output:
- A structured business insights file
- Segment-wise customer distribution
- Average Recency, Frequency, and Monetary values
- Actionable insights for marketing and retention teams

This report is designed for non-technical stakeholders and can be directly shared with business teams or clients.


In [ ]:
from pathlib import Path

def write_rfm_insights(df, output_path):
    total_customers = len(df)

    segment_counts = df["Segment"].value_counts()
    segment_percent = (segment_counts / total_customers * 100).round(2)

    avg_metrics = df.groupby("Segment")[["Recency", "Frequency", "Monetary"]].mean().round(2)

    lines = []
    lines.append("RFM CUSTOMER SEGMENTATION – BUSINESS INSIGHTS\n")
    lines.append(f"Total Customers Analyzed: {total_customers}\n")

    lines.append("Customer Distribution by Segment:\n")
    for seg in segment_counts.index:
        lines.append(
            f"- Segment {seg}: {segment_counts[seg]} customers ({segment_percent[seg]}%)"
        )

    lines.append("\nAverage RFM Metrics by Segment:\n")
    lines.append(avg_metrics.to_string())

    lines.append("\n\nKey Insights:\n")
    lines.append("- Segment A customers are the most valuable and recently active.")
    lines.append("- Segment B customers show moderate engagement and can be nurtured.")
    lines.append("- Segment C customers are low value or inactive and may need re-engagement campaigns.")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w") as f:
        f.write("\n".join(lines))


## Final Report Generation

In this step, we generate the final **RFM Business Insights Report**.

### What happens here:
- Uses the segmented RFM dataset
- Generates a structured business summary
- Saves insights to a text report for stakeholders

### Output:
- File: `outputs/reports/rfm_insights.txt`
- Contains:
  - Total customer count
  - Segment-wise distribution
  - Average RFM metrics
  - Business insights and recommendations

This report is designed for management, marketing, and decision-makers and serves as the final deliverable of the project.


In [ ]:
write_rfm_insights(
    rfm_labeled,
    "../outputs/reports/rfm_insights.txt"
)


## Final Output Validation

In this step, we verify that all expected project outputs have been successfully generated.

### What is checked:
- Cleaned transaction dataset  
- Final RFM table  
- All visualization charts  
- Business insights report  

### Purpose:
This validation ensures:
- The entire pipeline executed correctly
- No intermediate or final outputs are missing
- The project is ready for delivery or deployment

If all files exist, the pipeline is considered successfully completed.


In [ ]:
import os

required_outputs = [
    "../data/processed/cleaned_transactions.csv",
    "../data/processed/rfm_table.csv",
    "../outputs/charts/recency_distribution.png",
    "../outputs/charts/frequency_distribution.png",
    "../outputs/charts/monetary_distribution.png",
    "../outputs/charts/segment_counts.png",
    "../outputs/reports/rfm_insights.txt",
]

missing = [f for f in required_outputs if not os.path.exists(f)]

if not missing:
    print("✅ FINAL CHECK PASSED: All outputs generated successfully.")
else:
    print("❌ Missing files:")
    for f in missing:
        print(f)


## Revenue Contribution Analysis by Customer Segment

In this step, we analyze how much revenue each customer segment contributes to the business.

### What happens here:
- Calculate total revenue from all customers
- Group customers by segment
- Compute:
  - Number of customers per segment
  - Total revenue per segment
  - Percentage contribution to overall revenue

### Purpose:
This analysis helps identify:
- Which customer segments generate the most revenue
- High-value vs low-value customer groups
- Where marketing and retention efforts should be focused

The output is a summarized revenue table that supports business decision-making and strategy planning.


In [ ]:
import pandas as pd

# Total revenue
total_revenue = rfm_labeled["Monetary"].sum()

# Revenue contribution table
revenue_summary = (
    rfm_labeled
    .groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum")
    )
    .reset_index()
)

revenue_summary["Revenue %"] = (
    revenue_summary["Revenue"] / total_revenue * 100
).round(2)

revenue_summary
